In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive
!mkdir pressure_ulcer
%cd pressure_ulcer

/content/drive/MyDrive
mkdir: cannot create directory ‘pressure_ulcer’: File exists
/content/drive/MyDrive/pressure_ulcer


In [ ]:
import kagglehub
import os

# Download dataset (auto extracts)
path = kagglehub.dataset_download("leoscode/wound-segmentation-images")

print("Dataset path:", path)

In [ ]:
base_path = os.path.join(path, "data_wound_seg")
print("Base path:", base_path)

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")

    if level < 2:
        subindent = '  ' * (level + 1)
        for file in files[:5]:
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")

In [ ]:
# Install required libraries
!pip install segmentation-models-pytorch albumentations -q

In [ ]:
# Import all libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
import os

# Verify paired images and masks
# base_path = '/content/wound_dataset/data_wound_seg'
base_path = os.path.join(path, "data_wound_seg")

train_img_dir   = os.path.join(base_path, 'train_images')
train_mask_dir  = os.path.join(base_path, 'train_masks')
test_img_dir    = os.path.join(base_path, 'test_images')
test_mask_dir   = os.path.join(base_path, 'test_masks')

# Get sorted file lists
train_images = sorted(os.listdir(train_img_dir))
train_masks  = sorted(os.listdir(train_mask_dir))
test_images  = sorted(os.listdir(test_img_dir))
test_masks   = sorted(os.listdir(test_mask_dir))

print(f"Train images : {len(train_images)}")
print(f"Train masks  : {len(train_masks)}")
print(f"Test images  : {len(test_images)}")
print(f"Test masks   : {len(test_masks)}")

# Verify all names match
train_match = all(i == m for i, m in zip(train_images, train_masks))
test_match  = all(i == m for i, m in zip(test_images,  test_masks))

print(f"\nTrain pairs matched : {train_match}")
print(f"Test pairs matched  : {test_match}")

# Preview a few paired names
print("\nSample train pairs:")
for i in range(3):
    print(f"  {train_images[i]}  <-->  {train_masks[i]}")

In [ ]:
# Visualise sample image-mask pairs
def show_samples(img_dir, mask_dir, file_list, n=4):
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 4))
    axes[0, 0].set_title('Image', fontsize=14, fontweight='bold')
    axes[0, 1].set_title('Mask', fontsize=14, fontweight='bold')

    for i in range(n):
        img_path  = os.path.join(img_dir,  file_list[i])
        mask_path = os.path.join(mask_dir, file_list[i])

        img  = cv2.imread(img_path)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        axes[i, 0].imshow(img)
        axes[i, 0].axis('off')
        axes[i, 0].set_ylabel(file_list[i], fontsize=8)

        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].axis('off')

        # Print stats
        print(f"{file_list[i]} | Image shape: {img.shape} | "
              f"Mask shape: {mask.shape} | "
              f"Mask unique values: {np.unique(mask)}")

    plt.tight_layout()
    plt.show()

show_samples(train_img_dir, train_mask_dir, train_images, n=4)

In [ ]:
# Dataset class with augmentation
IMG_SIZE = 256

# Augmentation for training set
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# No augmentation for validation and test — only resize and normalize
val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class WoundDataset(Dataset):
    def __init__(self, img_dir, mask_dir, file_list, transform=None):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]

        # Load image
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load mask
        mask = cv2.imread(os.path.join(self.mask_dir, fname),
                          cv2.IMREAD_GRAYSCALE)

        # Binarise mask to 0 and 1
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img  = augmented['image']
            mask = augmented['mask']

        # Add channel dim to mask → (1, H, W)
        mask = mask.unsqueeze(0)

        return img, mask

print("WoundDataset class defined successfully!")

In [ ]:
# Split data and create DataLoaders
from torch.utils.data import DataLoader

# Split train → 70% train, 15% val  (test is already 15%)
train_files, val_files = train_test_split(
    train_images, test_size=0.177,  # 0.177 of 85% ≈ 15% of total
    random_state=42
)

print(f"Train samples      : {len(train_files)}")
print(f"Validation samples : {len(val_files)}")
print(f"Test samples       : {len(test_images)}")

# Create datasets
train_dataset = WoundDataset(train_img_dir, train_mask_dir,
                              train_files, transform=train_transform)
val_dataset   = WoundDataset(train_img_dir, train_mask_dir,
                              val_files,   transform=val_test_transform)
test_dataset  = WoundDataset(test_img_dir,  test_mask_dir,
                              test_images,  transform=val_test_transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=16,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# Quick sanity check on one batch
imgs, masks = next(iter(train_loader))
print(f"\nBatch image shape : {imgs.shape}")
print(f"Batch mask shape  : {masks.shape}")
print(f"Image dtype       : {imgs.dtype}")
print(f"Mask unique vals  : {torch.unique(masks)}")

In [ ]:
# Define all three models using segmentation_models_pytorch

# 1. U-Net (baseline)
unet_model = smp.Unet(
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,
    activation      = None
)

# 2. Attention U-Net
attention_unet_model = smp.UnetPlusPlus(
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,
    activation      = None
)

# 3. DeepLabV3+
deeplabv3plus_model = smp.DeepLabV3Plus(
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = 1,
    activation      = None
)

# Move all models to device
unet_model         = unet_model.to(device)
attention_unet_model = attention_unet_model.to(device)
deeplabv3plus_model  = deeplabv3plus_model.to(device)

# Print parameter counts
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"U-Net parameters             : {count_params(unet_model):,}")
print(f"Attention U-Net parameters   : {count_params(attention_unet_model):,}")
print(f"DeepLabV3+ parameters        : {count_params(deeplabv3plus_model):,}")
print(f"\nAll models loaded on: {device}")

In [ ]:
# Loss function, optimizer and metrics

# --- Loss Function: DiceBCE Loss ---
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceBCELoss, self).__init__()
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        # BCE part
        bce_loss = self.bce(preds, targets)

        # Dice part
        preds_sig = torch.sigmoid(preds)
        intersection = (preds_sig * targets).sum(dim=(2, 3))
        dice_loss = 1 - (2. * intersection + self.smooth) / (
            preds_sig.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth
        )
        dice_loss = dice_loss.mean()

        return bce_loss + dice_loss


# --- Metrics ---
def dice_coefficient(preds, targets, smooth=1e-6):
    preds    = torch.sigmoid(preds)
    preds    = (preds > 0.5).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    dice = (2. * intersection + smooth) / (
        preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + smooth
    )
    return dice.mean().item()

def iou_score(preds, targets, smooth=1e-6):
    preds    = torch.sigmoid(preds)
    preds    = (preds > 0.5).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    union        = (preds + targets - preds * targets).sum(dim=(2, 3))
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean().item()

def precision_score(preds, targets, smooth=1e-6):
    preds  = torch.sigmoid(preds)
    preds  = (preds > 0.5).float()
    tp = (preds * targets).sum(dim=(2, 3))
    fp = (preds * (1 - targets)).sum(dim=(2, 3))
    precision = (tp + smooth) / (tp + fp + smooth)
    return precision.mean().item()

def recall_score(preds, targets, smooth=1e-6):
    preds  = torch.sigmoid(preds)
    preds  = (preds > 0.5).float()
    tp = (preds * targets).sum(dim=(2, 3))
    fn = ((1 - preds) * targets).sum(dim=(2, 3))
    recall = (tp + smooth) / (tp + fn + smooth)
    return recall.mean().item()

def f1_score(preds, targets, smooth=1e-6):
    p = precision_score(preds, targets, smooth)
    r = recall_score(preds, targets, smooth)
    return 2 * p * r / (p + r + smooth)

def pixel_accuracy(preds, targets):
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()
    correct = (preds == targets).float().sum()
    total   = torch.numel(preds)
    return (correct / total).item()


# --- Instantiate loss ---
criterion = DiceBCELoss()

print("Loss function  : DiceBCE Loss")
print("Metrics defined: Dice, IoU, Precision, Recall, F1, Pixel Accuracy")
print("Criterion ready on device!")

In [ ]:
# Training and validation loop

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss, epoch_dice, epoch_iou = 0, 0, 0

    for imgs, masks in loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_dice += dice_coefficient(preds, masks)
        epoch_iou  += iou_score(preds, masks)

    n = len(loader)
    return epoch_loss/n, epoch_dice/n, epoch_iou/n


def validate(model, loader, criterion, device):
    model.eval()
    epoch_loss, epoch_dice, epoch_iou = 0, 0, 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss  = criterion(preds, masks)

            epoch_loss += loss.item()
            epoch_dice += dice_coefficient(preds, masks)
            epoch_iou  += iou_score(preds, masks)

    n = len(loader)
    return epoch_loss/n, epoch_dice/n, epoch_iou/n


def train_model(model, model_name, train_loader, val_loader,
                criterion, device, epochs=30, lr=1e-4):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', patience=4, factor=0.5)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_dice': [], 'val_dice': [],
        'train_iou' : [], 'val_iou' : []
    }

    best_val_loss  = float('inf')
    patience_count = 0
    early_stop_patience = 8

    print(f"\n{'='*55}")
    print(f"  Training: {model_name}")
    print(f"{'='*55}")

    for epoch in range(1, epochs + 1):

        train_loss, train_dice, train_iou = train_one_epoch(
            model, train_loader, optimizer, criterion, device)

        val_loss, val_dice, val_iou = validate(
            model, val_loader, criterion, device)

        scheduler.step(val_loss)

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(),
                       f'/content/{model_name}_best.pth')
            saved = '  *** saved best ***'
        else:
            saved = ''
            patience_count += 1

        print(f"Epoch [{epoch:02d}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | IoU: {train_iou:.4f} || "
              f"Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}"
              f"{saved}")

        # Early stopping
        if patience_count >= early_stop_patience:
            print(f"\nEarly stopping triggered at epoch {epoch}")
            break

    print(f"\nBest Val Loss: {best_val_loss:.4f}")
    print(f"Model saved to /content/{model_name}_best.pth")
    return history


print("Training loop defined successfully!")

In [ ]:
# Train U-Net
history_unet = train_model(
    model       = unet_model,
    model_name  = 'unet',
    train_loader= train_loader,
    val_loader  = val_loader,
    criterion   = criterion,
    device      = device,
    epochs      = 10,
    lr          = 1e-4
)


  Training: unet
Epoch [01/10] Train Loss: 1.4635 | Dice: 0.3735 | IoU: 0.2720 || Val Loss: 1.2181 | Dice: 0.6626 | IoU: 0.5268  *** saved best ***
Epoch [02/10] Train Loss: 1.1391 | Dice: 0.6825 | IoU: 0.5547 || Val Loss: 1.0669 | Dice: 0.7598 | IoU: 0.6396  *** saved best ***
Epoch [03/10] Train Loss: 1.0196 | Dice: 0.7194 | IoU: 0.6003 || Val Loss: 0.9631 | Dice: 0.7933 | IoU: 0.6852  *** saved best ***
Epoch [04/10] Train Loss: 0.9252 | Dice: 0.7391 | IoU: 0.6244 || Val Loss: 0.8757 | Dice: 0.7775 | IoU: 0.6662  *** saved best ***
Epoch [05/10] Train Loss: 0.8362 | Dice: 0.7591 | IoU: 0.6506 || Val Loss: 0.8183 | Dice: 0.7658 | IoU: 0.6546  *** saved best ***
Epoch [06/10] Train Loss: 0.7580 | Dice: 0.7775 | IoU: 0.6731 || Val Loss: 0.7355 | Dice: 0.8137 | IoU: 0.7139  *** saved best ***
Epoch [07/10] Train Loss: 0.6924 | Dice: 0.7856 | IoU: 0.6848 || Val Loss: 0.6736 | Dice: 0.8271 | IoU: 0.7300  *** saved best ***
Epoch [08/10] Train Loss: 0.6218 | Dice: 0.8011 | IoU: 0.7020 || 

In [ ]:
# Train Attention U-Net
history_attention_unet = train_model(
    model        = attention_unet_model,
    model_name   = 'attention_unet',
    train_loader = train_loader,
    val_loader   = val_loader,
    criterion    = criterion,
    device       = device,
    epochs       = 10,
    lr           = 1e-4
)


  Training: attention_unet


In [ ]:
# Train DeepLabV3+
history_deeplabv3plus = train_model(
    model        = deeplabv3plus_model,
    model_name   = 'deeplabv3plus',
    train_loader = train_loader,
    val_loader   = val_loader,
    criterion    = criterion,
    device       = device,
    epochs       = 15,
    lr           = 1e-4
)

In [ ]:
# Full test set evaluation with all metrics

def evaluate_on_test(model, model_name, test_loader,
                     criterion, device, weights_path):

    # Load best saved weights
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()

    total_loss, total_dice, total_iou      = 0, 0, 0
    total_prec, total_rec, total_f1, total_acc = 0, 0, 0, 0

    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)

            total_loss += criterion(preds, masks).item()
            total_dice += dice_coefficient(preds, masks)
            total_iou  += iou_score(preds, masks)
            total_prec += precision_score(preds, masks)
            total_rec  += recall_score(preds, masks)
            total_f1   += f1_score(preds, masks)
            total_acc  += pixel_accuracy(preds, masks)

    n = len(test_loader)
    results = {
        'Model'          : model_name,
        'Loss'           : round(total_loss / n, 4),
        'Dice'           : round(total_dice / n, 4),
        'IoU'            : round(total_iou  / n, 4),
        'Precision'      : round(total_prec / n, 4),
        'Recall'         : round(total_rec  / n, 4),
        'F1'             : round(total_f1   / n, 4),
        'Pixel Accuracy' : round(total_acc  / n, 4)
    }

    print(f"\n{'='*55}")
    print(f"  Test Results: {model_name}")
    print(f"{'='*55}")
    for k, v in results.items():
        if k != 'Model':
            print(f"  {k:<18}: {v}")

    return results


# Evaluate all three models
results_unet = evaluate_on_test(
    unet_model, 'U-Net',
    test_loader, criterion, device,
    '/content/unet_best.pth'
)

results_attention = evaluate_on_test(
    attention_unet_model, 'Attention U-Net',
    test_loader, criterion, device,
    '/content/attention_unet_best.pth'
)

results_deeplab = evaluate_on_test(
    deeplabv3plus_model, 'DeepLabV3+',
    test_loader, criterion, device,
    '/content/deeplabv3plus_best.pth'
)

In [ ]:
# Comparison table
import pandas as pd

all_results = [results_unet, results_attention, results_deeplab]
df = pd.DataFrame(all_results).set_index('Model')

print("\n" + "="*75)
print("  FINAL COMPARISON TABLE — TEST SET")
print("="*75)
print(df.to_string())
print("="*75)

# Highlight best value per metric
print("\n  Best model per metric:")
for col in df.columns:
    if col == 'Loss':
        best = df[col].idxmin()
    else:
        best = df[col].idxmax()
    print(f"  {col:<18}: {best} ({df.loc[best, col]})")

In [ ]:
# Plot training curves for all three models

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('Training Curves — All Models', fontsize=16, fontweight='bold')

models_history = [
    (history_unet,           'U-Net',           'blue'),
    (history_attention_unet, 'Attention U-Net',  'green'),
    (history_deeplabv3plus,  'DeepLabV3+',       'red')
]

metrics = [
    ('train_loss', 'val_loss', 'Loss'),
    ('train_dice', 'val_dice', 'Dice Coefficient'),
    ('train_iou',  'val_iou',  'IoU Score')
]

for row, (history, name, color) in enumerate(models_history):
    for col, (train_key, val_key, metric_name) in enumerate(metrics):
        ax = axes[row, col]
        epochs = range(1, len(history[train_key]) + 1)

        ax.plot(epochs, history[train_key],
                color=color, linestyle='--',
                label='Train', linewidth=2)
        ax.plot(epochs, history[val_key],
                color=color, linestyle='-',
                label='Val', linewidth=2)

        ax.set_title(f'{name} — {metric_name}', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric_name)
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to /content/training_curves.png")

In [ ]:
# Visual prediction comparison — all 3 models side by side

def visualise_predictions(models_dict, test_dataset, device, n_samples=6):

    fig, axes = plt.subplots(n_samples, 5, figsize=(20, n_samples * 4))

    col_titles = ['Image', 'Ground Truth',
                  'U-Net', 'Attention U-Net', 'DeepLabV3+']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=13, fontweight='bold')

    # Load best weights for all models
    for name, model in models_dict.items():
        path = f'/content/{name}_best.pth'
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()

    # Pick evenly spaced samples
    indices = np.linspace(0, len(test_dataset)-1, n_samples, dtype=int)

    for row, idx in enumerate(indices):
        img_tensor, mask_tensor = test_dataset[idx]

        # Original image (denormalize for display)
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_display = img_tensor.numpy().transpose(1, 2, 0)
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)

        # Ground truth
        gt = mask_tensor.squeeze().numpy()

        # Predictions from each model
        input_batch = img_tensor.unsqueeze(0).to(device)
        preds = []
        with torch.no_grad():
            for model in models_dict.values():
                pred = torch.sigmoid(model(input_batch))
                pred = (pred > 0.5).float()
                preds.append(pred.squeeze().cpu().numpy())

        # Plot
        axes[row, 0].imshow(img_display)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt, cmap='gray')
        axes[row, 1].axis('off')

        colors = ['Blues', 'Greens', 'Reds']
        for col, (pred, cmap) in enumerate(zip(preds, colors), start=2):
            axes[row, col].imshow(pred, cmap=cmap)
            axes[row, col].axis('off')

            # Compute per-sample dice for annotation
            pred_t  = torch.tensor(pred).unsqueeze(0).unsqueeze(0)
            gt_t    = mask_tensor.unsqueeze(0)
            p  = pred_t * gt_t
            d  = (2 * p.sum() + 1e-6) / (pred_t.sum() + gt_t.sum() + 1e-6)
            axes[row, col].set_xlabel(f'Dice: {d.item():.3f}',
                                       fontsize=10)

    plt.suptitle('Visual Comparison — Test Set Predictions',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('/content/visual_predictions.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to /content/visual_predictions.png")


models_dict = {
    'unet'           : unet_model,
    'attention_unet' : attention_unet_model,
    'deeplabv3plus'  : deeplabv3plus_model
}

visualise_predictions(models_dict, test_dataset, device, n_samples=6)

In [ ]:
# K-Fold Cross Validation (on training data)
from sklearn.model_selection import KFold

def run_kfold(model_fn, model_name, all_train_files,
              img_dir, mask_dir, device, k=5, epochs=5, lr=1e-4):

    kf      = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_results = []

    print(f"\n{'='*55}")
    print(f"  {k}-Fold Cross Validation: {model_name}")
    print(f"{'='*55}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_train_files), 1):

        fold_train = [all_train_files[i] for i in train_idx]
        fold_val   = [all_train_files[i] for i in val_idx]

        # Datasets
        fold_train_ds = WoundDataset(img_dir, mask_dir,
                                     fold_train, transform=train_transform)
        fold_val_ds   = WoundDataset(img_dir, mask_dir,
                                     fold_val,   transform=val_test_transform)

        fold_train_dl = DataLoader(fold_train_ds, batch_size=16,
                                   shuffle=True,  num_workers=2)
        fold_val_dl   = DataLoader(fold_val_ds,   batch_size=16,
                                   shuffle=False, num_workers=2)

        # Fresh model for each fold
        model     = model_fn().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = DiceBCELoss()

        best_dice = 0
        for epoch in range(1, epochs + 1):
            model.train()
            for imgs, masks in fold_train_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), masks)
                loss.backward()
                optimizer.step()

        # Evaluate fold
        model.eval()
        dice_scores, iou_scores = [], []
        with torch.no_grad():
            for imgs, masks in fold_val_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)
                dice_scores.append(dice_coefficient(preds, masks))
                iou_scores.append(iou_score(preds, masks))

        fold_dice = np.mean(dice_scores)
        fold_iou  = np.mean(iou_scores)
        fold_results.append({'dice': fold_dice, 'iou': fold_iou})

        print(f"  Fold {fold}/{k} — Dice: {fold_dice:.4f} | IoU: {fold_iou:.4f}")

    mean_dice = np.mean([r['dice'] for r in fold_results])
    std_dice  = np.std ([r['dice'] for r in fold_results])
    mean_iou  = np.mean([r['iou']  for r in fold_results])
    std_iou   = np.std ([r['iou']  for r in fold_results])

    print(f"\n  Mean Dice : {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"  Mean IoU  : {mean_iou:.4f} ± {std_iou:.4f}")

    return {'mean_dice': mean_dice, 'std_dice': std_dice,
            'mean_iou' : mean_iou,  'std_iou' : std_iou}


# Model factory functions (fresh weights each fold)
unet_fn      = lambda: smp.Unet(
    encoder_name='resnet34', encoder_weights='imagenet',
    in_channels=3, classes=1, activation=None)

attn_fn      = lambda: smp.UnetPlusPlus(
    encoder_name='resnet34', encoder_weights='imagenet',
    in_channels=3, classes=1, activation=None)

deeplab_fn   = lambda: smp.DeepLabV3Plus(
    encoder_name='resnet34', encoder_weights='imagenet',
    in_channels=3, classes=1, activation=None)

# Run cross validation — 5 folds, 5 epochs each
cv_unet   = run_kfold(unet_fn,    'U-Net',
                       train_images, train_img_dir, train_mask_dir, device)

cv_attn   = run_kfold(attn_fn,    'Attention U-Net',
                       train_images, train_img_dir, train_mask_dir, device)

cv_deeplab = run_kfold(deeplab_fn, 'DeepLabV3+',
                       train_images, train_img_dir, train_mask_dir, device)

In [ ]:
# Final summary report plot

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Pressure Ulcer Segmentation — Final Results Summary',
             fontsize=16, fontweight='bold')

models      = ['U-Net', 'Attention\nU-Net', 'DeepLabV3+']
colors      = ['#2196F3', '#4CAF50', '#F44336']

# --- Plot 1: Test Set Metrics Bar Chart ---
ax1 = axes[0, 0]
metrics_names  = ['Dice', 'IoU', 'Precision', 'Recall', 'F1']
unet_scores    = [0.7883, 0.6902, 0.7432, 0.9234, 0.8223]
attn_scores    = [0.7960, 0.7033, 0.7474, 0.9416, 0.8321]
deeplab_scores = [0.8041, 0.7068, 0.7813, 0.8988, 0.8345]

x     = np.arange(len(metrics_names))
width = 0.25

ax1.bar(x - width, unet_scores,    width, label='U-Net',
        color=colors[0], alpha=0.85)
ax1.bar(x,          attn_scores,   width, label='Attention U-Net',
        color=colors[1], alpha=0.85)
ax1.bar(x + width,  deeplab_scores, width, label='DeepLabV3+',
        color=colors[2], alpha=0.85)

ax1.set_title('Test Set Performance Metrics', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names)
ax1.set_ylabel('Score')
ax1.set_ylim(0.6, 1.0)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)


# --- Plot 2: Cross Validation Dice with error bars ---
ax2 = axes[0, 1]
cv_means = [cv_unet['mean_dice'],
            cv_attn['mean_dice'],
            cv_deeplab['mean_dice']]
cv_stds  = [cv_unet['std_dice'],
            cv_attn['std_dice'],
            cv_deeplab['std_dice']]

bars = ax2.bar(models, cv_means, color=colors, alpha=0.85,
               yerr=cv_stds, capsize=8, error_kw={'linewidth': 2})
ax2.set_title('5-Fold Cross Validation — Dice ± Std', fontweight='bold')
ax2.set_ylabel('Mean Dice Score')
ax2.set_ylim(0.6, 0.95)
ax2.grid(axis='y', alpha=0.3)

for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + std + 0.005,
             f'{mean:.4f}\n±{std:.4f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')


# --- Plot 3: Radar / Spider Chart ---
ax3 = axes[1, 0]
ax3.set_title('Model Comparison — Test Dice vs CV Dice', fontweight='bold')

test_dice = [0.7883, 0.7960, 0.8041]
cv_dice   = [cv_unet['mean_dice'],
             cv_attn['mean_dice'],
             cv_deeplab['mean_dice']]

x_pos = np.arange(len(models))
ax3.plot(x_pos, test_dice, 'o--', color='navy',
         linewidth=2, markersize=8, label='Test Dice')
ax3.plot(x_pos, cv_dice,   's--', color='darkgreen',
         linewidth=2, markersize=8, label='CV Dice')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(['U-Net', 'Attention U-Net', 'DeepLabV3+'])
ax3.set_ylabel('Dice Score')
ax3.set_ylim(0.75, 0.86)
ax3.legend()
ax3.grid(True, alpha=0.3)

for i, (td, cd) in enumerate(zip(test_dice, cv_dice)):
    ax3.annotate(f'{td:.4f}', (i, td),
                 textcoords='offset points', xytext=(0, 10),
                 ha='center', fontsize=9, color='navy')
    ax3.annotate(f'{cd:.4f}', (i, cd),
                 textcoords='offset points', xytext=(0, -15),
                 ha='center', fontsize=9, color='darkgreen')


# --- Plot 4: Summary Table ---
ax4 = axes[1, 1]
ax4.axis('off')

table_data = [
    ['Metric',        'U-Net',  'Attn U-Net', 'DeepLabV3+'],
    ['Test Dice',     '0.7883', '0.7960',     '0.8041'],
    ['Test IoU',      '0.6902', '0.7033',     '0.7068'],
    ['Precision',     '0.7432', '0.7474',     '0.7813'],
    ['Recall',        '0.9234', '0.9416',     '0.8988'],
    ['F1 Score',      '0.8223', '0.8321',     '0.8345'],
    ['Pixel Acc.',    '0.9957', '0.9960',     '0.9967'],
    ['CV Dice',       '0.7905', '0.8203',     '0.7923'],
    ['CV Std',        '±0.0357','±0.0255',    '±0.0049'],
]

table = ax4.table(cellText=table_data[1:],
                  colLabels=table_data[0],
                  cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.3, 1.8)

# Header styling
for col in range(4):
    table[0, col].set_facecolor('#37474F')
    table[0, col].set_text_props(color='white', fontweight='bold')

# Row styling
row_colors = ['#E3F2FD', '#E8F5E9', '#FFEBEE']
for row in range(1, len(table_data)):
    for col in range(4):
        if col == 0:
            table[row, col].set_facecolor('#ECEFF1')
            table[row, col].set_text_props(fontweight='bold')
        else:
            table[row, col].set_facecolor(row_colors[col-1])

ax4.set_title('Complete Results Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('/content/final_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Final summary saved to /content/final_summary.png")